# MSD_15to1_validation — Litinski 15-to-1 の正当性・計数の検証

| ID | 内容 |
|---|---|
| M1 | 式(2) の三直交性、式(4) が式(2) と（行変形＋列の並べ替えで）同値であること、式(4)→式(1)（puncturing）、式(1) → Fig. 15 の 11 回転（図からの手転記と照合） |
| M2 | 理想入力 → 測定結果は決定的に +1、出力 $T^\dagger|+\rangle$（[Lit] "actually $T^\dagger$"） |
| M3 | 15 量子ビット符号（Fig. 14）：transversal $T$ = 論理 $T^\dagger$、5 量子ビット回路と同じ出力 |
| M4 | 誤り検出：1・2個の $Z$ 誤りは全て検出、3個で未検出 35 通り（$35p^3$）、未検出の 35 通りは全て誤った出力 |
| M5 | selective π/4（Fig. 17a）と auto-corrected π/8（Fig. 17b）：選んだ 4 種の Pauli 積とランダム入力について、全測定分岐で Pauli 補正のみで正しい |
| M6 | tile 数 11、time step 11、$121d^3$ の再計数（2通りの数え方） |
| M7 | 物理レベルの格子手術（stim）：2 パッチ＋1 tile の合併で $\bar Z_1\bar Z_2$ が測定される；量子ビット増分 |
| M8 | 回転表面符号パッチの量子ビット数・CNOT 数の閉形式、tile↔物理の換算の一貫性 |
| M9 | **Fig. 17c の 11 個の多パッチ測定 $P\otimes Z_m$ の物理レベル構成**：$\mathbb F_2$ による厳密判定（$d=3,5$ は合併符号の距離も MILP、$d=7$）、stim による測定値の照合（$d=3,5$）、高速版との一致 |

In [1]:
# ==== 環境設定（Google Colab / ローカル共通）====
# このノートブックは単独で実行できる（共通コードは下に埋め込まれており、他のファイルを読み込まない）。
# 出力（results/, figures/）はこのランタイムの作業ディレクトリに保存される。Colab ではランタイム終了で
# 消えるので、残したい場合は Google Drive を使う：
#   from google.colab import drive; drive.mount('/content/drive')
#   os.makedirs('/content/drive/MyDrive/D4_vs_MSD', exist_ok=True); os.chdir('/content/drive/MyDrive/D4_vs_MSD')
import os, sys, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
RESULTS, FIGURES = 'results', 'figures'
os.makedirs(RESULTS, exist_ok=True); os.makedirs(FIGURES, exist_ok=True)
SEED = 20261024
print('cwd =', os.getcwd(), '| Colab =', IN_COLAB)

cwd = /tmp/claude-0/-home-user-2026-10-24-StudyWithClaude/fee9d03f-2fbf-502f-b670-8966176acf54/scratchpad/run3/msd | Colab = False


In [2]:
# ==== 外部ライブラリ（Colab に無いものだけをインストール）====
import importlib.util
for _mod, _pkg in [('stim', 'stim'), ('scipy', 'scipy'), ('sympy', 'sympy'), ('pandas', 'pandas'),
                   ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg], check=True)
import numpy as np, pandas as pd, scipy
print('numpy', np.__version__, '| scipy', scipy.__version__, '| pandas', pd.__version__)

numpy 2.5.3 | scipy 1.18.1 | pandas 3.0.5


## 共通ライブラリ（埋め込み・関数定義のみ）
Google Colab ではノートブックごとに別のランタイムで実行されるため、他のファイルを読み込まずに済むよう、共通コードをこのノートブックに埋め込んでいる（各ノートの共通コードは同一のソースから自動生成されており、内容は一致する）。
**このセクションは上から順に実行するだけでよく、編集する必要はない**（Colab では見出しの左の三角で折りたためる）。

### 共通ライブラリ（F2 線形代数・ゲート／ガジェット・スケジューラ・Clifford+T 分解・計数ルール R1–R3）

In [3]:
"""common_lib: F2 linear algebra, gate/gadget model, scheduler, decompositions, cost model rules."""
import itertools, math
import numpy as np
from collections import Counter, defaultdict

In [4]:
# ---------------------------------------------------------------- F2 algebra
def f2_rref(M):
    """Row-reduce a uint8 matrix over F2. Returns (R, pivot_cols)."""
    M = (np.array(M, dtype=np.uint8) & 1).copy()
    rows, cols = M.shape
    piv = []
    r = 0
    for c in range(cols):
        if r >= rows:
            break
        nz = np.nonzero(M[r:, c])[0]
        if len(nz) == 0:
            continue
        p = r + nz[0]
        if p != r:
            M[[r, p]] = M[[p, r]]
        others = np.nonzero(M[:, c])[0]
        others = others[others != r]
        if len(others):
            M[others] ^= M[r]
        piv.append(c)
        r += 1
    return M[:r], piv


def f2_rank(M):
    if len(M) == 0:
        return 0
    return len(f2_rref(M)[1])


def f2_nullspace(M, n=None):
    """Basis (rows) of {x : M x = 0} over F2."""
    M = np.array(M, dtype=np.uint8)
    if M.size == 0:
        return np.eye(n, dtype=np.uint8)
    n = M.shape[1]
    Rm, piv = f2_rref(M)
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        v = np.zeros(n, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if Rm[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis, dtype=np.uint8).reshape(len(basis), n)


def f2_in_rowspace(Rref, piv, v):
    """Is v in the row space of the RREF matrix Rref (pivots piv)?"""
    v = np.array(v, dtype=np.uint8) & 1
    w = v.copy()
    for i, p in enumerate(piv):
        if w[p]:
            w ^= Rref[i]
    return not w.any()


def f2_solve(A, b):
    """Solve x A = b (x row vector, A rows = generators). Returns x or None."""
    A = np.array(A, dtype=np.uint8)
    m, n = A.shape
    aug = np.concatenate([A, np.eye(m, dtype=np.uint8)], axis=1)
    Rm, piv = f2_rref(aug)
    w = np.concatenate([np.array(b, dtype=np.uint8) & 1, np.zeros(m, dtype=np.uint8)])
    for i, p in enumerate(piv):
        if p >= n:
            break
        if w[p]:
            w ^= Rm[i]
    if w[:n].any():
        return None
    return w[n:]




import itertools
from collections import Counter, defaultdict

ARITY = {'INIT_X': 1, 'INIT_Z': 1, 'MEAS_X': 1, 'MEAS_Z': 1, 'H': 1, 'X': 1, 'Z': 1, 'S': 1, 'SDG': 1,
         'T': 1, 'TDG': 1, 'CNOT': 2, 'CZ': 2, 'CS': 2, 'CSDG': 2, 'CCZ': 3}
NONCLIFFORD = {'T', 'TDG', 'CS', 'CSDG', 'CCZ'}

In [5]:
# ------------------------------------------------------------------ minimum-weight logical (MILP)
def min_weight_odd(Hc, lvec, cols):
    """min |x| over x in F2^cols with Hc x = 0 and lvec . x = 1 (exact MILP, HiGHS)."""
    from scipy.optimize import milp, LinearConstraint, Bounds
    H = np.array(Hc, dtype=float)[:, cols]
    H = H[np.any(H, axis=1)]
    l = np.array(lvec, dtype=float)[cols]
    m, n = H.shape
    # variables: x (n, binary), y (m, integer), z (1, integer)
    nv = n + m + 1
    c = np.concatenate([np.ones(n), np.zeros(m + 1)])
    A1 = np.concatenate([H, -2 * np.eye(m), np.zeros((m, 1))], axis=1)
    A2 = np.concatenate([l[None, :], np.zeros((1, m)), -2 * np.ones((1, 1))], axis=1)
    cons = [LinearConstraint(A1, 0, 0), LinearConstraint(A2, 1, 1)]
    integrality = np.ones(nv)
    ub = np.concatenate([np.ones(n), np.full(m + 1, np.inf)])
    res = milp(c, constraints=cons, integrality=integrality, bounds=Bounds(np.zeros(nv), ub))
    assert res.status == 0, res.message
    x = np.round(res.x[:n]).astype(np.uint8)
    out = np.zeros(len(lvec), dtype=np.uint8)
    out[cols] = x
    return int(x.sum()), out

In [6]:
# ------------------------------------------------------------------ gadgets
def pauli_check_gadget(anc, xq=(), zq=()):
    """Measure X(xq) Z(zq) with ancilla anc (standard CNOT circuit)."""
    g = [('INIT_X', anc)]
    g += [('CNOT', anc, q) for q in sorted(xq)]           # ancilla controls -> X on data
    g += [('CZ', anc, q) for q in sorted(zq)]             # controlled Z on data
    g += [('MEAS_X', anc)]
    return g


def z_check_gadget(anc, zq):
    g = [('INIT_Z', anc)] + [('CNOT', q, anc) for q in sorted(zq)] + [('MEAS_Z', anc)]
    return g


def clifford_x_gadget(anc, parts):
    """Measure a product of X-type Clifford operators S = S_1 S_2 ... with
    S_k = X(s_k) prod CZ(pairs_k): ancilla |+>, controlled-S (= CNOTs from anc and
    CCZ(anc,i,j) for each CZ pair; rightmost factor first), measure X.
    parts: list of (xqubits, czpairs) in operator order S_1 S_2 ..."""
    g = [('INIT_X', anc)]
    for xq, pairs in reversed(parts):
        g += [('CCZ', anc, i, j) for (i, j) in pairs]      # CZ part acts first within S_k
        g += [('CNOT', anc, q) for q in sorted(xq)]
    g += [('MEAS_X', anc)]
    return g

In [7]:
# ------------------------------------------------------------------ scheduling
def asap_layers(gates):
    """Greedy ASAP layering of a gate list respecting qubit order (no reordering of gates that
    share a qubit).  Returns list of layers (lists of gates)."""
    ready = defaultdict(int)
    layers = []
    for gt in gates:
        qs = gt[1:]
        t = max(ready[q] for q in qs)
        while len(layers) <= t:
            layers.append([])
        layers[t].append(gt)
        for q in qs:
            ready[q] = t + 1
    return layers


def gadget_support(gad):
    s = set()
    for gt in gad:
        s |= set(gt[1:])
    return s


def disjoint_groups(gadgets):
    """Greedy colouring: partition gadgets so that gadgets in a group touch disjoint qubits.
    Measurements within a group are then exactly simultaneous and the sequential-group
    schedule is valid for commuting (on the relevant subspace) generators."""
    groups = []          # list of (used_qubits, [gadgets])
    order = sorted(range(len(gadgets)), key=lambda i: -len(gadget_support(gadgets[i])))
    for i in order:
        sup = gadget_support(gadgets[i])
        for used, members in groups:
            if not (used & sup):
                used |= sup
                members.append(gadgets[i])
                break
        else:
            groups.append((set(sup), [gadgets[i]]))
    return [m for _, m in groups]


def layer_duration(layer, dur):
    return max(dur[g[0]] for g in layer)


def circuit_time(layers, dur):
    return sum(layer_duration(l, dur) for l in layers)


def round_schedule(gadgets, dur):
    """Upper-bound schedule of one round: disjoint groups executed sequentially, each group
    ASAP-layered.  Returns dict with depth (layers), time (weighted by dur), groups."""
    groups = disjoint_groups(gadgets)
    depth = 0
    time = 0
    for grp in groups:
        gl = asap_layers([gt for gad in grp for gt in gad])
        depth += len(gl)
        time += circuit_time(gl, dur)
    return {'groups': len(groups), 'depth_ub': depth, 'time_ub': time}


def round_lower_bound(gadgets, dur):
    """Lower bound on the duration of one round: every qubit executes its gates sequentially,
    so time >= max_q sum of durations of gates acting on q."""
    busy = defaultdict(float)
    cnt = defaultdict(int)
    for gad in gadgets:
        for gt in gad:
            for q in gt[1:]:
                busy[q] += dur[gt[0]]
                cnt[q] += 1
    return {'depth_lb': max(cnt.values()), 'time_lb': max(busy.values())}


def gate_counts(gates):
    c = Counter(g[0] for g in gates)
    return dict(c)

In [8]:
# ------------------------------------------------------------------ decompositions
# Explicit Clifford+T decompositions (verified numerically in the validation notebook).
def ccz_to_clifford_t(a, b, c):
    """CCZ with 7 T/T^dag and 6 CNOT (standard Toffoli decomposition, Nielsen & Chuang Fig. 4.9,
    with the target Hadamards removed)."""
    return [('CNOT', b, c), ('TDG', c), ('CNOT', a, c), ('T', c), ('CNOT', b, c), ('TDG', c),
            ('CNOT', a, c), ('T', b), ('T', c), ('CNOT', a, b), ('T', a), ('TDG', b), ('CNOT', a, b)]


def cs_to_clifford_t(a, b):
    """CS = diag(1,1,1,i) = T_a T_b CNOT(a,b) T^dag_b CNOT(a,b)."""
    return [('T', a), ('T', b), ('CNOT', a, b), ('TDG', b), ('CNOT', a, b)]


def csdg_to_clifford_t(a, b):
    return [('TDG', a), ('TDG', b), ('CNOT', a, b), ('T', b), ('CNOT', a, b)]


def decompose(gates):
    out = []
    for g in gates:
        if g[0] == 'CCZ':
            out += ccz_to_clifford_t(*g[1:])
        elif g[0] == 'CS':
            out += cs_to_clifford_t(*g[1:])
        elif g[0] == 'CSDG':
            out += csdg_to_clifford_t(*g[1:])
        else:
            out.append(g)
    return out


DUR_UNIT = {k: 1 for k in ARITY}            # every primitive = 1 time unit (ASSUMPTION, parametrised)

In [9]:
# ------------------------------------------------- commuting-class scheduler
DIAG_GATES = {'CZ', 'CCZ', 'CS', 'CSDG', 'T', 'TDG', 'S', 'SDG', 'Z'}


def gate_roles(gt):
    """Per-qubit role of an entangling/1q unitary: 'Z' (diagonal in Z basis), 'X' (X-type)."""
    name = gt[0]
    if name in DIAG_GATES:
        return {q: 'Z' for q in gt[1:]}
    if name == 'CNOT':
        return {gt[1]: 'Z', gt[2]: 'X'}
    if name == 'X':
        return {gt[1]: 'X'}
    return {q: '?' for q in gt[1:]}      # H, INIT, MEAS: never shared between gadgets


def gadget_roles(gad):
    roles = {}
    for gt in gad:
        for q, r in gate_roles(gt).items():
            if roles.get(q, r) != r:
                roles[q] = '?'
            else:
                roles[q] = r
    return roles


def commuting_classes(gadgets, n_orders=6, seed=0):
    """Best (fewest classes) of several greedy colourings (different gadget orders)."""
    import random
    best = None
    orders = [list(range(len(gadgets)))]
    orders.append(sorted(orders[0], key=lambda i: tuple(sorted(set(gadget_roles(gadgets[i]).values())))))
    rnd = random.Random(seed)
    for _ in range(n_orders):
        o = orders[0][:]
        rnd.shuffle(o)
        orders.append(o)
    for o in orders:
        cl = _commuting_classes_greedy([gadgets[i] for i in o])
        if best is None or len(cl) < len(best):
            best = cl
    return best


def _commuting_classes_greedy(gadgets):
    """Greedy colouring of the gadget conflict graph.  Two gadgets are compatible if on every
    shared qubit both act with the same role ('Z' or 'X'); then all their gates commute and the
    joint circuit is equivalent to measuring them one after another (sufficient condition)."""
    roles = [gadget_roles(g) for g in gadgets]
    classes = []     # list of (role_map, [idx])
    for i, rm in enumerate(roles):
        placed = False
        for cmap, members in classes:
            ok = all(cmap.get(q, r) == r and r != '?' for q, r in rm.items() if q in cmap)
            if ok:
                for q, r in rm.items():
                    cmap[q] = r
                members.append(i)
                placed = True
                break
        if not placed:
            classes.append((dict(rm), [i]))
    return [[gadgets[i] for i in m] for _, m in classes]


def class_layers(gadgets):
    """Layering of one commuting class: init layer, greedy layering of the (mutually commuting)
    entangling gates, measurement layer."""
    inits = [g for gad in gadgets for g in gad if g[0].startswith('INIT')]
    meas = [g for gad in gadgets for g in gad if g[0].startswith('MEAS')]
    body = [g for gad in gadgets for g in gad if not (g[0].startswith('INIT') or g[0].startswith('MEAS'))]
    # greedy: most-constrained gates first
    body = sorted(body, key=lambda g: -len(g))
    layers = []
    busy = []
    for g in body:
        qs = set(g[1:])
        for li, b in enumerate(busy):
            if not (b & qs) and layers[li][0][0] == g[0]:     # keep one gate type per layer
                b |= qs
                layers[li].append(g)
                break
        else:
            busy.append(set(qs))
            layers.append([g])
    return ([inits] if inits else []) + layers + ([meas] if meas else [])


def round_schedule_classes(gadgets, dur):
    classes = commuting_classes(gadgets)
    depth = 0
    time = 0
    for c in classes:
        ls = class_layers(c)
        depth += len(ls)
        time += circuit_time(ls, dur)
    return {'classes': len(classes), 'depth_ub': depth, 'time_ub': time}


import math
from collections import Counter

SOURCE_LIT = 'literature'
SOURCE_COUNT = 'counted (explicit circuit)'
SOURCE_ASSUME = 'assumption (parameter)'
SOURCE_UNRES = 'unresolved (excluded / illustrative)'

DEFAULT_PARAMS = dict(
    kappa0=1.0,      # FSC |+_L> preparation rounds = ceil(kappa0*d)   [ASSUMPTION; Kobayashi: not specified]
    kappa_in=1.0,    # switch-in rounds            = ceil(kappa_in*d)  [ASSUMPTION; Kobayashi: O(d)]
    kappa_out=1.0,   # FSC EC rounds after return  = ceil(kappa_out*d) [ASSUMPTION; Kobayashi: O(d)]
    tau_rounds=1.0,  # Litinski: 1 time step = tau_rounds*d code cycles [LITERATURE: d cycles]
    t_inj_FT=6,      # Litinski App. A (Li's injection): 3 + 3 code cycles, d-independent [LITERATURE]
    t_inj_NL=1,      # noiseless-minimal injection: 1 round [ASSUMPTION]
)


def rounds_for(model, kappa, d):
    if model == 'NL_min':
        return 1
    return int(math.ceil(kappa * d))

In [10]:
# ----------------------------------------------------------------- summaries
def summarize(rows):
    tot = Counter()
    for r in rows:
        for k, v in r['gates'].items():
            tot[k] += v
    peak = max(r['qubits'] for r in rows)
    rounds = sum(r['rounds'] for r in rows)
    tl = sum(r['time_lb'] for r in rows if r['time_lb'] is not None)
    tu = sum(r['time_ub'] for r in rows if r['time_ub'] is not None)
    v_round = sum(r['qubits'] * r['rounds'] for r in rows)
    v_tl = sum(r['qubits'] * r['time_lb'] for r in rows if r['time_lb'] is not None)
    v_tu = sum(r['qubits'] * r['time_ub'] for r in rows if r['time_ub'] is not None)
    nonclif = {k: v for k, v in tot.items() if k in NONCLIFFORD}
    tcount = 7 * tot.get('CCZ', 0) + 3 * (tot.get('CS', 0) + tot.get('CSDG', 0)) + tot.get('T', 0) + tot.get('TDG', 0)
    return dict(peak_qubits=peak, rounds=rounds, time_lb=tl, time_ub=tu, V_qubit_round=v_round,
                V_qubit_time_lb=v_tl, V_qubit_time_ub=v_tu, V_peak_x_rounds_UB=peak * rounds,
                gates=dict(tot), nonclifford=nonclif, T_count_if_decomposed=tcount)


def uncounted_costs(protocol, comparison):
    """Costs that are known to exist but are not included in the totals (listed, never summed as 0)."""
    out = [('classical decoding / control latency', 'both comparisons, both protocols')]
    if protocol == 'Kobayashi_D4':
        out.append(('JIT-decoder classical processing', 'not modelled'))
        if comparison == 'B':
            out.append(('FSC -> standard rotated surface-code patch conversion', 'UNRESOLVED: no protocol constructed'))
    else:
        out.append(('twist-based Z_m (x) Y_0 seam checks between |m> and |0>', 'O(d) measurement qubits, not constructed'))
    return out

In [11]:
# ------------------------------------------------------------------ table helpers
def jsonify_columns(df, cols=('gates', 'nonclifford')):
    """Store dict-valued columns as JSON strings (same format as the saved CSV files)."""
    import json
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(lambda g: g if isinstance(g, str) else json.dumps({k: float(v) for k, v in g.items()}))
    return df


def identify_polynomial(values, npts=4):
    """Interpolate a polynomial in d through the first npts points (degree <= npts-1) and report
    whether it reproduces ALL given points exactly.  values: dict d -> integer.  Returns (expr, ok)."""
    import sympy
    x = sympy.Symbol('d')
    ds = sorted(values)
    if len(ds) <= npts:
        return None, False
    p = sympy.interpolate([(dd, int(round(values[dd]))) for dd in ds[:npts]], x)
    ok = all(sympy.nsimplify(p.subs(x, dd) - int(round(values[dd]))) == 0 for dd in ds)
    return sympy.expand(p), ok

### 15-to-1 ライブラリ（行列・状態ベクトル回路・Fig. 17c 配置・多パッチ格子手術の明示的構成・資源モデル）

In [12]:
"""Litinski 'A Game of Surface Codes' Sec. 3: 15-to-1 distillation.
Logical-level circuits (statevector) + tile-level layout model + physical (patch-level) model."""
import itertools
import numpy as np

In [13]:
# ----------------------------------------------------------------- matrices (transcribed)
# Eq. (1) of Litinski: rows 1-4 = X stabilizers, row 5 = logical X, columns = 15 qubits
M15 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)
# Eq. (2): triorthogonal matrix G ; Eq. (4): its row-echelon form G~
G_TRI = np.array([[1] * 16,
                  [0] * 8 + [1] * 8,
                  [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
                  [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
                  [0, 1] * 8], dtype=np.uint8)
G_TILDE = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1],
    [0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1],
    [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1]], dtype=np.uint8)


def triorthogonal(G):
    G = G.astype(int)
    a = all(G[i].sum() % 8 == 0 for i in range(len(G)))
    b = all((G[i] * G[j]).sum() % 4 == 0 for i, j in itertools.combinations(range(len(G)), 2))
    c = all((G[i] * G[j] * G[k]).sum() % 2 == 0 for i, j, k in itertools.combinations(range(len(G)), 3))
    return a, b, c

In [14]:
# ----------------------------------------------------------------- statevector helpers
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
PAULI = {'I': I2, 'X': X, 'Y': Y, 'Z': Z}
KET0 = np.array([1, 0], dtype=complex)
KETP = np.array([1, 1], dtype=complex) / np.sqrt(2)
T_GATE = np.diag([1, np.exp(1j * np.pi / 4)])
KET_M = T_GATE @ KETP                 # |m> = |0> + e^{i pi/4}|1>  (normalised)


def kron_all(ops):
    out = np.array([[1]], dtype=complex)
    for o in ops:
        out = np.kron(out, o)
    return out


def pauli_on(n, paulistr_by_qubit):
    """Dense Pauli operator; paulistr_by_qubit: dict qubit -> 'X'|'Y'|'Z'.  Qubit 0 = most significant."""
    return kron_all([PAULI[paulistr_by_qubit.get(i, 'I')] for i in range(n)])


def apply_1q(psi, n, q, U):
    psi = psi.reshape([2] * n)
    psi = np.moveaxis(np.tensordot(U, psi, axes=([1], [q])), 0, q)
    return psi.reshape(-1)


def z_rotation_diag(n, qubits, phi):
    """Diagonal of exp(-i phi Z_{q1} Z_{q2} ...)."""
    idx = np.arange(2 ** n)
    par = np.zeros(2 ** n, dtype=int)
    for q in qubits:
        par ^= (idx >> (n - 1 - q)) & 1
    return np.exp(-1j * phi * (1 - 2 * par))


def measure_pauli(psi, P, rng=None, forced=None):
    """Projective measurement of Hermitian Pauli matrix P.  Returns (outcome +-1, post-state, prob)."""
    Ppsi = P @ psi
    p_plus = np.real(np.vdot(psi, psi + Ppsi)) / 2
    if forced is None:
        out = 1 if rng.random() < p_plus else -1
    else:
        out = forced
    post = (psi + out * Ppsi) / 2
    prob = p_plus if out == 1 else 1 - p_plus
    if prob > 1e-12:
        post = post / np.sqrt(prob)
    return out, post, prob


def equal_up_to_phase(a, b, tol=1e-9):
    ov = np.vdot(a, b)
    return abs(abs(ov) - np.linalg.norm(a) * np.linalg.norm(b)) < tol

In [15]:
# ----------------------------------------------------------------- 15-to-1, 5-qubit circuit (Fig. 15)
def fig15_rotations():
    """The 11 Z-type pi/8 rotations of Fig. 15 = columns 5..15 of M15 (rows = qubits 1..5).
    Columns 1..4 (weight-1 on qubits 4,3,2,1) are absorbed into the |m> initial states."""
    rots = []
    for j in range(4, 15):
        rots.append(tuple(int(i) for i in np.nonzero(M15[:, j])[0]))
    return rots


def run_15to1_5qubit(inputs=None, phi=np.pi / 8, rng=None):
    """Simulate Fig. 15.  inputs: list of 4 single-qubit states for qubits 1-4 (default ideal |m>).
    Returns dict outcome-tuple -> (probability, output state of qubit 5)."""
    n = 5
    if inputs is None:
        inputs = [KET_M] * 4
    psi = kron_all([s.reshape(2, 1) for s in inputs] + [KETP.reshape(2, 1)]).reshape(-1)
    for qs in fig15_rotations():
        psi = z_rotation_diag(n, qs, phi) * psi
    # measure qubits 1-4 in X basis: enumerate all outcomes
    res = {}
    for outs in itertools.product((1, -1), repeat=4):
        st = psi.copy()
        prob = 1.0
        for q, o in enumerate(outs):
            _, st, p = measure_pauli(st, pauli_on(n, {q: 'X'}), forced=o)
            prob *= p
            if p < 1e-14:
                break
        if prob < 1e-14:
            res[outs] = (0.0, None)
            continue
        # reduced state of qubit 5 (qubits 1-4 are in X eigenstates -> product)
        st = st.reshape(16, 2)
        # project onto the X-eigenstates of qubits 1..4
        xs = [np.array([1, o], dtype=complex) / np.sqrt(2) for o in outs]
        bra = kron_all([x.reshape(1, 2) for x in xs]).reshape(-1)
        out5 = bra.conj() @ st
        out5 = out5 / np.linalg.norm(out5)
        res[outs] = (prob, out5)
    return res

In [16]:
# ----------------------------------------------------------------- 15-qubit code check (Fig. 14)
def rm15_logical_check(gate='T'):
    """Prepare |+_L> of the 15-qubit code (X stabilisers = rows 1-4 of M15, logical X = row 5,
    Z stabilisers = F2-orthogonal complement), apply transversal T (or T^dag) and return the
    logical expectation values <X_L>, <Y_L-like> to identify the logical gate."""
    n = 15
    from_rows = M15.astype(int)
    # Z stabilisers: nullspace of M15 over F2
    Hz = f2_nullspace_simple(M15)
    dim = 2 ** n
    psi = np.zeros(dim, dtype=complex)
    psi[0] = 1.0
    idx = np.arange(dim)
    # |+_L> = prod (1+X_s)/2 (1+X_L)/2 |0..0>  (|0..0> is +1 eigenstate of all Z stabilisers)
    for row in from_rows:
        mask = int(''.join(str(b) for b in row), 2)
        psi = (psi + psi[idx ^ mask]) / 2
    psi /= np.linalg.norm(psi)
    # transversal gate
    w = np.array([bin(i).count('1') for i in idx])
    ph = np.exp(1j * np.pi / 4 * w) if gate == 'T' else np.exp(-1j * np.pi / 4 * w)
    psi = ph * psi
    # logical operators: X_L = X(row5), Z_L = Z on a vector z with z.row5 = 1, z.rows1-4 = 0
    xl = int(''.join(str(b) for b in from_rows[4]), 2)
    zvec = find_logical_z(M15)
    zl = int(''.join(str(b) for b in zvec), 2)
    par = np.array([bin(i & zl).count('1') % 2 for i in idx])
    Zl_psi = (1 - 2 * par) * psi
    Xl_psi = psi[idx ^ xl]
    # Y_L = i X_L Z_L
    Yl_psi = 1j * Zl_psi[idx ^ xl]
    ex = np.real(np.vdot(psi, Xl_psi))
    ey = np.real(np.vdot(psi, Yl_psi))
    ez = np.real(np.vdot(psi, Zl_psi))
    # stabiliser expectation values (X stabilisers should be lost only up to the Clifford
    # correction; Z stabilisers preserved)
    zst = []
    for hz in Hz:
        m = int(''.join(str(b) for b in hz), 2)
        par = np.array([bin(i & m).count('1') % 2 for i in idx])
        zst.append(np.real(np.vdot(psi, (1 - 2 * par) * psi)))
    return dict(X_L=ex, Y_L=ey, Z_L=ez, min_Zstab=min(zst), n_Zstab=len(Hz))


def f2_nullspace_simple(M):
    M = M.copy() % 2
    rows, cols = M.shape
    A = M.copy()
    piv = []
    r = 0
    for c in range(cols):
        p = next((i for i in range(r, rows) if A[i, c]), None)
        if p is None:
            continue
        A[[r, p]] = A[[p, r]]
        for i in range(rows):
            if i != r and A[i, c]:
                A[i] ^= A[r]
        piv.append(c)
        r += 1
    free = [c for c in range(cols) if c not in piv]
    basis = []
    for f in free:
        v = np.zeros(cols, dtype=np.uint8)
        v[f] = 1
        for i, p in enumerate(piv):
            if A[i, f]:
                v[p] = 1
        basis.append(v)
    return np.array(basis)


def find_logical_z(M):
    """z with M[0:4] z = 0, M[4] z = 1 (brute force over weight)."""
    for w in range(1, 16):
        for sup in itertools.combinations(range(15), w):
            z = np.zeros(15, dtype=np.uint8)
            z[list(sup)] = 1
            s = (M.astype(int) @ z) % 2
            if not s[:4].any() and s[4] == 1:
                return z
    raise RuntimeError

In [17]:
# ----------------------------------------------------------------- auto-corrected pi/8 (Fig. 17b)
def auto_corrected_pi8(P_data, psi_data, rng=None):
    """Fig. 17b on k data qubits + |m> + |0>.  P_data: dict qubit->Pauli on the data.
    Enumerates all measurement branches; for each, returns the Pauli correction needed and
    whether the post-correction data state equals exp(-i pi/8 P)|psi>.
    Measurement sequence (Litinski Fig. 17b):
      1. measure P (x) Z_m  and  Z_m (x) Y_0   (commuting, simultaneous)
      2. if P(x)Z_m = +1: measure |0>-qubit in Z ; else in X     (selective pi/4, Fig. 17a)
      3. measure |m> in X
      4. Pauli correction P^{c} with c determined by the outcomes (P_{pi/2} = P up to phase)."""
    k = int(np.log2(len(psi_data)))
    n = k + 2
    m, a = k, k + 1
    psi = np.kron(np.kron(psi_data, KET_M), KET0)
    P1 = pauli_on(n, {**P_data, m: 'Z'})
    P2 = pauli_on(n, {m: 'Z', a: 'Y'})
    Pd = pauli_on(k, P_data)
    target = np.cos(np.pi / 8) * psi_data - 1j * np.sin(np.pi / 8) * (Pd @ psi_data)
    branches = []
    for o1 in (1, -1):
        for o2 in (1, -1):
            s1, st, p1 = measure_pauli(psi, P1, forced=o1)
            if p1 < 1e-14:
                continue
            s2, st, p2 = measure_pauli(st, P2, forced=o2)
            if p2 < 1e-14:
                continue
            basis = 'Z' if o1 == 1 else 'X'
            for o3 in (1, -1):
                s3, st3, p3 = measure_pauli(st, pauli_on(n, {a: basis}), forced=o3)
                if p3 < 1e-14:
                    continue
                for o4 in (1, -1):
                    s4, st4, p4 = measure_pauli(st3, pauli_on(n, {m: 'X'}), forced=o4)
                    if p4 < 1e-14:
                        continue
                    # extract data state: ancillas are in known product eigenstates
                    anc_m = np.array([1, o4], dtype=complex) / np.sqrt(2)
                    anc_a = (np.array([1, 0], dtype=complex) if o3 == 1 else np.array([0, 1], dtype=complex)) \
                        if basis == 'Z' else np.array([1, o3], dtype=complex) / np.sqrt(2)
                    bra = np.kron(anc_m, anc_a)
                    dat = st4.reshape(2 ** k, 4) @ bra.conj()
                    dat /= np.linalg.norm(dat)
                    ok0 = equal_up_to_phase(dat, target)
                    ok1 = equal_up_to_phase(Pd @ dat, target)
                    branches.append(dict(PZ=o1, ZY=o2, anc_basis=basis, anc=o3, Xm=o4,
                                         prob=p1 * p2 * p3 * p4,
                                         correction='I' if ok0 else ('P' if ok1 else 'FAIL')))
    return branches

In [18]:
# ----------------------------------------------------------------- tile layout (Fig. 17c)
# Grid of 3 rows x 5 columns (row 0 = top).  Each entry: label of the patch occupying the tile.
FIG17C_STEP1 = [
    [None, None, 'q1', 'q3', None],
    [None, None, None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_ROTATION = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', 'anc', 'anc', 'q5'],
    [None, None, 'q2', 'q4', None]]
FIG17C_FINAL = [
    ['zero', 'm', 'q1', 'q3', None],
    ['zero', 'm', None, None, 'q5'],
    [None, None, 'q2', 'q4', None]]
# NOTE: in Step 1 the 2x2 block on the left is shaded as part of the block (reserved) and the
# |m> states for the rotations are prepared there; Litinski counts 11 tiles for the block.


def tile_schedule(n_rot=11):
    """List of (label, tau, layout) for the Fig. 17c protocol.  tau = time steps (1 = d cycles)."""
    sched = [('step1_init', 0, FIG17C_STEP1)]
    for r in range(n_rot):
        sched.append((f'rotation_{r + 1}', 1, FIG17C_ROTATION))
        sched.append((f'readout_{r + 1}', 0, FIG17C_FINAL))
    sched.append(('final_X_measurement', 0, FIG17C_FINAL))
    return sched


def tiles_used(sched):
    used = set()
    for _, _, lay in sched:
        for i, row in enumerate(lay):
            for j, lab in enumerate(row):
                if lab is not None:
                    used.add((i, j))
    return used


import numpy as np


def rotated_checks(rows, cols, origin=(0, 0)):
    """Stabilizers of a rotated planar code on data qubits (r,c), 0<=r<rows, 0<=c<cols (shifted by
    origin).  Check type from global parity (r+c)%2 of the plaquette corner -> consistent tiling.
    Weight-2 Z checks on top/bottom, weight-2 X checks on left/right, so that the logical Z
    string runs vertically, along the left/right boundaries (the boundaries used for Z(x)Z surgery)."""
    r0, c0 = origin
    checks = []
    for r in range(rows + 1):
        for c in range(cols + 1):
            qs = [(r0 + rr, c0 + cc) for rr in (r - 1, r) for cc in (c - 1, c)
                  if 0 <= rr < rows and 0 <= cc < cols]
            typ = 'X' if (r0 + r + c0 + c) % 2 == 0 else 'Z'
            if len(qs) == 4:
                checks.append((typ, tuple(qs)))
            elif len(qs) == 2:
                horizontal_edge = (r == 0 or r == rows)
                if (horizontal_edge and typ == 'Z') or ((c == 0 or c == cols) and typ == 'X'):
                    checks.append((typ, tuple(qs)))
    return checks


def patch_gadgets(checks, qindex, anc_start):
    gads = []
    a = anc_start
    for typ, qs in checks:
        idx = [qindex[q] for q in qs]
        if typ == 'X':
            gads.append(pauli_check_gadget(a, xq=idx))
        else:
            gads.append(z_check_gadget(a, idx))
        a += 1
    return gads


def patch_qubit_count(d):
    ch = rotated_checks(d, d)
    return d * d, len(ch)


def lattice_surgery_zz_check(d, gap=None, seed=0):
    """Two d x d rotated patches separated by `gap` columns (default d, i.e. one tile).
    Prepare both in |+_L>, perform one noiseless merge round on the full rectangle, and verify
    with stim that  Zbar1 Zbar2  becomes deterministic and equals the product of the merge-round
    outcomes of an F2 combination of merged Z checks.  Returns a dict of results."""
    import stim
    if gap is None:
        gap = d
    W = 2 * d + gap
    data = [(r, c) for r in range(d) for c in range(W)]
    qi = {q: i for i, q in enumerate(data)}
    p1 = rotated_checks(d, d, (0, 0))
    p2 = rotated_checks(d, d, (0, d + gap))
    merged = rotated_checks(d, W, (0, 0))
    nq = len(data)

    def vec(qs):
        v = np.zeros(nq, dtype=np.uint8)
        for q in qs:
            v[qi[q]] ^= 1
        return v

    def logical_z(checks, cols):
        # Z string along a column that commutes with the patch's X checks
        X = [vec(qs) for t, qs in checks if t == 'X']
        for c in cols:
            z = vec([(r, c) for r in range(d)])
            if all((x.astype(int) @ z) % 2 == 0 for x in X):
                return z
        raise RuntimeError('no column logical')

    z1 = logical_z(p1, range(0, d))
    z2 = logical_z(p2, range(d + gap, W))
    O = z1 ^ z2
    sim = stim.TableauSimulator()
    sim.set_num_qubits(nq)
    rng = np.random.default_rng(seed)
    # |+> on all data of both patches and on the gap (Litinski Fig. 41: ancilla region in |+>)
    for i in range(nq):
        sim.h(i)

    def meas(typ, qs):
        ps = stim.PauliString(nq)
        for q in qs:
            ps[qi[q]] = typ
        return sim.measure_observable(ps)

    pre = {}
    for t, qs in p1 + p2:
        pre[(t, qs)] = meas(t, qs)
    ps = stim.PauliString(nq)
    for i in np.nonzero(O)[0]:
        ps[int(i)] = 'Z'
    before = sim.peek_observable_expectation(ps)
    out = {}
    for t, qs in merged:
        out[(t, qs)] = meas(t, qs)
    after = sim.peek_observable_expectation(ps)
    Zm = [(k, vec(k[1])) for k in merged if k[0] == 'Z']
    A = np.array([v for _, v in Zm])
    coef = f2_solve(A, O)
    parity = 1
    for c, (k, _) in zip(coef, Zm):
        if c:
            parity *= -1 if out[k] else 1
    n_checks_merged = len(merged)
    return dict(d=d, gap=gap, data_qubits=nq, merged_checks=n_checks_merged,
                total_qubits=nq + n_checks_merged,
                exp_before=before, exp_after=after, product_of_outcomes=parity,
                consistent=(after == parity), O_in_span=coef is not None)

In [19]:
# ----------------------------------------------------------------- Litinski (15-to-1)

In [20]:
# ---------------------------------------------------------- Fig. 17c multi-patch lattice surgery (physical level)
def _tile_rect(r, c, d, h=1, w=1):
    return (r * d, (r + h) * d, c * d, (c + w) * d)


def fig17_layout(d):
    """Patches of the rotation step of [Lit] Fig. 17c on a grid of d x d-data-qubit tiles
    (3 tile rows x 5 tile columns, row 0 = top).  Each entry: (rect, rule) with rect =
    (r0, r1, c0, c1) data-qubit ranges and rule(side, r, c) -> 'X'|'Z' = type of the weight-2
    boundary checks on that side.  For every data patch the boundary facing the ancilla strip is of
    'X-check' type, so that its logical Z string runs along it ([Lit] 'Z edge')."""
    tb = lambda side, r, c: 'X' if side in ('top', 'bottom') else 'Z'       # q1..q4 (merge via top/bottom)
    lr = lambda side, r, c: 'X' if side in ('left', 'right') else 'Z'       # q5 (merge via its left edge)
    mr = _tile_rect(0, 1, d, h=2)

    def m_rule(side, r, c, r0=mr[0]):
        # |m> occupies two tiles; its corners are moved ([Lit] Fig. 2b) so that only the lower half of
        # its right edge (facing the ancilla strip) is a Z edge: segments Z(top+upper right), X(lower
        # right), Z(bottom), X(left).
        if side == 'right':
            return 'X' if r >= r0 + d else 'Z'
        return 'X' if side == 'left' else 'Z'
    return {'q1': (_tile_rect(0, 2, d), tb), 'q3': (_tile_rect(0, 3, d), tb),
            'q2': (_tile_rect(2, 2, d), tb), 'q4': (_tile_rect(2, 3, d), tb),
            'q5': (_tile_rect(1, 4, d), lr), 'm': (mr, m_rule)}


def _owner_map(patches, anc_rect=None):
    own = {}
    for name, ((r0, r1, c0, c1), _) in patches.items():
        for r in range(r0, r1):
            for c in range(c0, c1):
                own[(r, c)] = name
    if anc_rect:
        r0, r1, c0, c1 = anc_rect
        for r in range(r0, r1):
            for c in range(c0, c1):
                own[(r, c)] = 'anc'
    return own


def _side_of(rect, r, c):
    r0, r1, c0, c1 = rect
    if r == r0: return 'top'
    if r == r1: return 'bottom'
    if c == c0: return 'left'
    if c == c1: return 'right'
    return None


def patch_checks(name, rect, rule):
    """Checks (type, qubits) of one standalone rotated patch with the given boundary rule."""
    r0, r1, c0, c1 = rect
    out = []
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            typ = 'X' if (r + c) % 2 == 0 else 'Z'
            qs = [(rr, cc) for rr in (r - 1, r) for cc in (c - 1, c) if r0 <= rr < r1 and c0 <= cc < c1]
            if len(qs) == 4:
                out.append((typ, tuple(qs)))
            elif len(qs) == 2 and typ == rule(_side_of(rect, r, c), r, c):
                out.append((typ, tuple(qs)))
    return out


def surgery_new_checks(d, merged, anc_free='Z', w3=('Z',), seam_end=('Z',), corner1=('Z',)):
    """Checks newly measured when the ancilla strip (tiles (1,2),(1,3), all qubits prepared in |+>)
    is merged with the patches in `merged` ([Lit] App. A, Fig. 41): every plaquette that touches an
    ancilla qubit, restricted to the merged region.  Truncated plaquettes: weight 3 kept if its type is
    in w3; weight 2 on the free ancilla boundary kept if its type == anc_free; weight 2 straddling
    ancilla/patch kept if its type is in seam_end; an ancilla corner qubit left without a weight-2 check
    (both neighbouring edges free) is measured in Z if the plaquette type is in corner1."""
    pats = fig17_layout(d)
    anc_rect = _tile_rect(1, 2, d, w=2)
    own = _owner_map(pats, anc_rect)
    region = {q for q, o in own.items() if o == 'anc' or o in set(merged)}
    r0, r1, c0, c1 = anc_rect
    N = []
    for r in range(r0, r1 + 1):
        for c in range(c0, c1 + 1):
            typ = 'X' if (r + c) % 2 == 0 else 'Z'
            qs = [(rr, cc) for rr in (r - 1, r) for cc in (c - 1, c) if (rr, cc) in region]
            if not any(own[q] == 'anc' for q in qs):
                continue
            if len(qs) == 4:
                N.append((typ, tuple(qs)))
            elif len(qs) == 1 and typ in corner1:
                N.append((typ, tuple(qs)))           # isolated ancilla corner qubit: measured in Z
            elif len(qs) == 3 and typ in w3:
                N.append((typ, tuple(qs)))
            elif len(qs) == 2:
                if all(own[q] == 'anc' for q in qs):
                    af = anc_free(_side_of(anc_rect, r, c), r, c) if callable(anc_free) else anc_free
                    if typ == af:
                        N.append((typ, tuple(qs)))
                elif typ in seam_end:
                    N.append((typ, tuple(qs)))
    return N, pats, anc_rect, own


def _vec(qs, idx, n):
    v = np.zeros(n, dtype=np.uint8)
    for q in qs:
        v[idx[q]] ^= 1
    return v


def _mats(checks, idx, n):
    X = np.array([_vec(qs, idx, n) for t, qs in checks if t == 'X'], dtype=np.uint8).reshape(-1, n)
    Z = np.array([_vec(qs, idx, n) for t, qs in checks if t == 'Z'], dtype=np.uint8).reshape(-1, n)
    return X, Z


def _rank(M):
    return f2_rank(M) if len(M) else 0


def _span_dim_intersection(A, B):
    """dim(span A  intersect  span B)."""
    if len(A) == 0 or len(B) == 0:
        return 0
    return _rank(A) + _rank(B) - _rank(np.vstack([A, B]))


def commutant(G, Nopp):
    """Basis of {g in span(G) : g . v = 0 for all rows v of Nopp}, plus the indices of the rows of G
    that individually commute (kept as measured checks)."""
    if len(G) == 0:
        return np.zeros((0, Nopp.shape[1]), np.uint8), []
    Bm = (G.astype(int) @ Nopp.T.astype(int)) % 2 if len(Nopp) else np.zeros((len(G), 0), int)
    keep = [i for i in range(len(G)) if not Bm[i].any()]
    if Bm.shape[1] == 0:
        return G.copy(), keep
    C = f2_nullspace(Bm.T.astype(np.uint8), len(G))
    return (C.astype(int) @ G.astype(int) % 2).astype(np.uint8), keep


def css_logical_bases(SX, SZ, n):
    """Independent logical representatives (modulo stabilizers) of a CSS code."""
    def indep(base, cands):
        out = []
        cur = base if len(base) else np.zeros((0, n), np.uint8)
        r = _rank(cur)
        for v in cands:
            nxt = np.vstack([cur, v[None, :]])
            rn = _rank(nxt)
            if rn > r:
                out.append(v); cur = nxt; r = rn
        return out
    Lz = indep(SZ, list(f2_nullspace(SX, n)) if len(SX) else list(np.eye(n, dtype=np.uint8)))
    Lx = indep(SX, list(f2_nullspace(SZ, n)) if len(SZ) else list(np.eye(n, dtype=np.uint8)))
    return Lx, Lz


def verify_multipatch(d, merged, distance=False, **opts):
    """Exact (F2) verification that measuring the new checks N implements the logical measurement of
    O = prod_{p in merged} Zbar_p on the Fig. 17c layout and nothing else.
      V1 N mutually commute;  V2 every standalone patch has k = 1 (distance d if distance=True);
      V3 O in span(N_Z) + span(G0_Z) and exactly one independent Z-logical is measured;
      V4 no X-logical is measured;
      V5 every logical commuting with O (all Zbar_p, Xbar_p for p not merged, Xbar_p Xbar_q for merged
         pairs) has a representative (mod G0) commuting with N;
      V6 (distance=True) merged code G1 = <N, commutant of N in G0> has k = 5 and distance >= d.
    Also returns the physically measured generating set during the merged rounds."""
    N, pats, anc_rect, own = surgery_new_checks(d, merged, **opts)
    qs = sorted(own); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
    G0 = []
    for name, (rect, rule) in pats.items():
        G0 += patch_checks(name, rect, rule)
    G0X, G0Z = _mats(G0, idx, n)
    anc_q = [idx[q] for q in qs if own[q] == 'anc']
    G0Xa = np.vstack([G0X, np.eye(n, dtype=np.uint8)[anc_q]])          # ancilla |+> : single-qubit X
    NX, NZ = _mats(N, idx, n)
    res = {'n_data': n, 'n_new_checks': len(N)}
    res['V1_new_checks_commute'] = not ((NX.astype(int) @ NZ.T.astype(int)) % 2).any()
    # standalone logicals
    Zb, Xb, ok2, dists = {}, {}, True, {}
    for name, (rect, rule) in pats.items():
        ch = patch_checks(name, rect, rule)
        sub = sorted({q for _, c in ch for q in c} | {(r, c) for r in range(rect[0], rect[1]) for c in range(rect[2], rect[3])})
        sidx = {q: i for i, q in enumerate(sub)}
        SX, SZ = _mats(ch, sidx, len(sub))
        Lx, Lz = css_logical_bases(SX, SZ, len(sub))
        ok2 &= len(Lz) == 1 and len(Lx) == 1
        emb = lambda v: _vec([sub[i] for i in np.nonzero(v)[0]], idx, n)
        Zb[name], Xb[name] = emb(Lz[0]), emb(Lx[0])
        if distance:
            dz = min_weight_odd(SX, Lx[0], list(range(len(sub))))[0]
            dx = min_weight_odd(SZ, Lz[0], list(range(len(sub))))[0]
            dists[name] = (dx, dz)
    res['V2_patches_k1'] = ok2
    if distance:
        res['V2_patch_distances'] = dists
        res['V2_patch_distance_ok'] = all(min(v) >= d for v in dists.values())
    names = list(pats)
    O = np.zeros(n, np.uint8)
    for p in merged:
        O ^= Zb[p]
    AZ = np.vstack([G0Z, NZ])
    res['V3_O_determined'] = _rank(np.vstack([AZ, O[None, :]])) == _rank(AZ)
    ZL = np.array([Zb[p] for p in names])
    res['V3_num_Z_logicals_measured'] = _span_dim_intersection(AZ, np.vstack([G0Z, ZL])) - _rank(G0Z)
    AX = np.vstack([G0Xa, NX])
    XL = np.array([Xb[p] for p in names])
    res['V4_num_X_logicals_measured'] = _span_dim_intersection(AX, np.vstack([G0Xa, XL])) - _rank(G0Xa)

    def has_rep(L, G, Nopp):
        if len(Nopp) == 0:
            return True
        lhs = (Nopp.astype(int) @ L.astype(int)) % 2
        if not lhs.any():
            return True
        Bm = (Nopp.astype(int) @ G.T.astype(int)) % 2
        return f2_solve(Bm.T.astype(np.uint8), lhs.astype(np.uint8)) is not None
    ok5 = all(has_rep(Zb[p], G0Z, NX) for p in names)
    ml = sorted(merged)
    xs = [Xb[p] for p in names if p not in merged] + [Xb[a] ^ Xb[b] for a, b in zip(ml, ml[1:])]
    ok5 &= all(has_rep(v, G0Xa, NZ) for v in xs)
    res['V5_commuting_logicals_preserved'] = ok5
    # merged-phase measured generating set: N + old checks that commute with N + products for the rest
    CZ_, keepZ = commutant(G0Z, NX)
    CX_, keepX = commutant(G0X, NZ)
    G1X = np.vstack([NX, CX_]); G1Z = np.vstack([NZ, CZ_])
    extraZ = _rank(np.vstack([NZ, G0Z[keepZ]] if keepZ else [NZ, np.zeros((0, n), np.uint8)]))
    res['n_old_checks_kept'] = len(keepZ) + len(keepX)
    res['n_old_checks_combined'] = (_rank(G1Z) - _rank(np.vstack([NZ] + ([G0Z[keepZ]] if keepZ else []))) +
                                    _rank(G1X) - _rank(np.vstack([NX] + ([G0X[keepX]] if keepX else []))))
    k1 = n - _rank(G1X) - _rank(G1Z)
    res['merged_k'] = k1
    res['valid'] = (res['V1_new_checks_commute'] and ok2 and res['V3_O_determined'] and
                    res['V3_num_Z_logicals_measured'] == 1 and res['V4_num_X_logicals_measured'] == 0 and ok5
                    and k1 == len(names) - 1)
    if distance and res['valid']:
        Lx, Lz = css_logical_bases(G1X, G1Z, n)
        dz = min(min_weight_odd(G1X, lx, list(range(n)))[0] for lx in Lx)
        dx = min(min_weight_odd(G1Z, lz, list(range(n)))[0] for lz in Lz)
        res['V6_merged_distance'] = (dx, dz)
        res['valid'] &= min(dx, dz) >= d
    res['_data'] = dict(N=N, G0=G0, own=own, idx=idx, pats=pats, anc_rect=anc_rect, G1X=G1X, G1Z=G1Z,
                        keepX=keepX, keepZ=keepZ, CX=CX_, CZ=CZ_, G0X=G0X, G0Z=G0Z, NX=NX, NZ=NZ, O=O)
    return res


def rotation_measured_checks(d, merged, res=None):
    """Explicit list of checks measured in every round of the merged phase of one rotation:
    the new checks N, the old patch checks that commute with N, and (if needed) products of old checks
    completing the commutant (local, weight <= 8).  Returns (checks, res)."""
    res = res or verify_multipatch(d, merged)
    D = res['_data']
    qs = sorted(D['own']); n = len(qs)
    out = list(D['N'])
    G0 = D['G0']
    G0Xc = [c for c in G0 if c[0] == 'X']; G0Zc = [c for c in G0 if c[0] == 'Z']
    out += [G0Xc[i] for i in D['keepX']] + [G0Zc[i] for i in D['keepZ']]
    for typ, C, base in [('X', D['CX'], np.vstack([D['NX']] + ([D['G0X'][D['keepX']]] if D['keepX'] else []))),
                         ('Z', D['CZ'], np.vstack([D['NZ']] + ([D['G0Z'][D['keepZ']]] if D['keepZ'] else [])))]:
        cur = base
        for v in sorted(C, key=lambda v: int(v.sum())):
            nxt = np.vstack([cur, v[None, :]])
            if _rank(nxt) > _rank(cur):
                supp = tuple(qs[i] for i in np.nonzero(v)[0])
                assert len(supp) <= 8, 'non-local combined generator'
                out.append((typ, supp)); cur = nxt
    return out, res


def stim_multipatch_check(d, merged, seed=0):
    """Noiseless stim run: patches in |+_L> (data |+>, one round of their checks), ancilla strip in |+>,
    then one round of the merged-phase checks.  Checks that O = prod Zbar is random before and
    deterministic after, with value equal to the product of the merged-round outcomes of an F2
    combination of measured Z checks (times recorded pre-merge Z-check outcomes)."""
    import stim
    checks, res = rotation_measured_checks(d, merged)
    D = res['_data']
    qs = sorted(D['own']); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
    sim = stim.TableauSimulator(seed=seed)
    sim.set_num_qubits(n)
    for i in range(n):
        sim.h(i)
    def meas(t, cq):
        ps = stim.PauliString(n)
        for q in cq: ps[idx[q]] = t
        return sim.measure_observable(ps)
    pre = [(t, cq, meas(t, cq)) for t, cq in D['G0']]
    Ops = stim.PauliString(n)
    for i in np.nonzero(D['O'])[0]: Ops[int(i)] = 'Z'
    before = sim.peek_observable_expectation(Ops)
    post = [(t, cq, meas(t, cq)) for t, cq in checks]
    after = sim.peek_observable_expectation(Ops)
    rows = [(_vec(cq, idx, n), o) for t, cq, o in post + pre if t == 'Z']
    A = np.array([r[0] for r in rows]); coef = f2_solve(A, D['O'])
    pred = 1
    for c_, (_, o) in zip(coef, rows):
        if c_: pred *= -1 if o else 1
    return dict(before=before, after=after, predicted=pred, consistent=(after == pred) and before == 0)


def rotation_measured_checks_fast(d, merged):
    """Same measured set as rotation_measured_checks, computed without the global verification:
    old checks that commute with all new checks are kept; among the (few, seam-adjacent) old checks
    that anticommute, the commuting combinations are found by a small nullspace computation.
    (Equality with the fully verified construction is checked in the validation notebook.)"""
    N, pats, anc_rect, own = surgery_new_checks(d, merged)
    G0 = []
    for name, (rect, rule) in pats.items():
        G0 += patch_checks(name, rect, rule)
    out = list(N)
    for typ, opp in [('X', 'Z'), ('Z', 'X')]:
        olds = [c for c in G0 if c[0] == typ]
        news = [set(c[1]) for c in N if c[0] == opp]
        anti = []
        for c in olds:
            s = set(c[1])
            if any(len(s & nn) % 2 for nn in news):
                anti.append(c)
            else:
                out.append(c)
        if anti:
            qset = sorted({q for c in anti for q in c[1]} | {q for nn in news for q in nn})
            li = {q: i for i, q in enumerate(qset)}
            Aold = np.array([_vec(c[1], li, len(qset)) for c in anti])
            Nn = np.array([_vec(nn, li, len(qset)) for nn in news if nn & set(li)])
            C, _ = commutant(Aold, Nn)
            cur = np.zeros((0, len(qset)), np.uint8)
            for v in sorted(C, key=lambda v: int(v.sum())):
                nxt = np.vstack([cur, v[None, :]])
                if _rank(nxt) > _rank(cur):
                    out.append((typ, tuple(qset[i] for i in np.nonzero(v)[0]))); cur = nxt
    return out, pats, own


class MSDCostData:
    """d-dependent counts of the Litinski 15-to-1 block (Fig. 17c).
    * tile model: Litinski's accounting (computed in msd_stages).
    * physical model: explicit construction of every rotation's merged phase (surgery_new_checks /
      rotation_measured_checks_fast): data qubits of q1..q5, |m> (2 tiles, moved corners), the ancilla
      strip (2 tiles) and the |0> patch (2 tiles), one measurement qubit per measured check, CNOTs per
      round and round time from the shared scheduler.  NOT constructed: the twist-based Z_m (x) Y_0
      surgery between |m> and |0> (its extra seam checks are not counted)."""

    def __init__(self, d, dur=None, explicit=True):
        dur = dur or DUR_UNIT
        self.d = d
        self.dur = dur
        ch = rotated_checks(d, d)
        qi = {(r, c): r * d + c for r in range(d) for c in range(d)}
        g = patch_gadgets(ch, qi, d * d)
        ub = round_schedule_classes(g, dur)
        lb = round_lower_bound(g, dur)
        self.round = dict(time_lb=lb['time_lb'], time_ub=ub['time_ub'])       # one isolated patch
        self.sq_patch = d * d + len(ch)                                       # 2d^2 - 1
        ch2 = rotated_checks(d, 2 * d)
        self.tall_patch = 2 * d * d + len(ch2)                                # 4d^2 - 1
        self.cnot_sq = sum(len(qs) for _, qs in ch)
        self.cnot_tall = sum(len(qs) for _, qs in ch2)
        ch3 = rotated_checks(d, 3 * d)
        self.gap_tile = 3 * d * d + len(ch3) - 2 * self.sq_patch               # 2-patch Z(x)Z merge increment
        if not explicit:
            return
        pats = fig17_layout(d)
        idle_checks = []
        for name, (rect, rule) in pats.items():
            idle_checks += patch_checks(name, rect, rule)
        zero_rect = (0, 2 * d, -d, 0)                                         # |0> patch: tiles (0,0),(1,0)
        zero_checks = patch_checks('zero', zero_rect, lambda side, r, c: 'X' if side in ('left', 'right') else 'Z')
        def cost(checks, n_data):
            coords = sorted({q for _, qq in checks for q in qq})
            qi_ = {q: i for i, q in enumerate(coords)}
            gads = []
            for j, (t, qq) in enumerate(checks):
                idx_ = [qi_[q] for q in qq]
                gads.append(pauli_check_gadget(10 ** 6 + j, xq=idx_) if t == 'X' else z_check_gadget(10 ** 6 + j, idx_))
            return dict(data=n_data, checks=len(checks), qubits=n_data + len(checks),
                        cnot=sum(len(qq) for _, qq in checks),
                        time_lb=round_lower_bound(gads, dur)['time_lb'],
                        time_ub=round_schedule_classes(gads, dur)['time_ub'])
        n_patch_data = 7 * d * d                     # q1..q5 + |m> (2 tiles)
        self.idle = cost(idle_checks + zero_checks, n_patch_data + 2 * d * d)
        self.rot = []
        for r_ in fig15_rotations():
            P = tuple(sorted({'q%d' % (i + 1) for i in r_})) + ('m',)
            chk, _, _ = rotation_measured_checks_fast(d, P)
            self.rot.append(dict(P=P, **cost(chk + zero_checks, n_patch_data + 2 * d * d + 2 * d * d)))


def msd_stages(md, model, params=DEFAULT_PARAMS, comparison='A', level='physical', data_only=False):
    """level='tile': Litinski's accounting (tiles x d^2 data qubits, or x 2d^2 incl. measurement
    qubits; time = rounds x isolated-patch round time).  level='physical': explicit per-rotation
    construction (see MSDCostData)."""
    d = md.d
    tau = rounds_for(model, params['tau_rounds'], d)
    rows = []
    if level == 'tile':
        per_tile = d * d if data_only else 2 * d * d
        rot = [dict(qubits=11 * per_tile, cnot=None, time_lb=md.round['time_lb'], time_ub=md.round['time_ub'])] * 11
        idle = dict(qubits=9 * per_tile, time_lb=md.round['time_lb'], time_ub=md.round['time_ub'])
        src_rot = SOURCE_LIT
    else:
        key = 'data' if data_only else 'qubits'
        rot = [dict(qubits=r[key], cnot=r['cnot'], time_lb=r['time_lb'], time_ub=r['time_ub']) for r in md.rot]
        idle = dict(qubits=md.idle[key], time_lb=md.idle['time_lb'], time_ub=md.idle['time_ub'])
        src_rot = SOURCE_COUNT
    meas_layer = md.dur['MEAS_X']

    def add(stage, qubits, rounds, rt, extra_time, gates, source, note=''):
        rows.append(dict(protocol='Litinski_15to1', comparison=comparison, model=model, d=d, stage=stage,
                         qubits=qubits, rounds=rounds,
                         time_lb=rounds * rt['time_lb'] + extra_time,
                         time_ub=rounds * rt['time_ub'] + extra_time,
                         gates=dict(gates), source=source, note=note))

    inj = 0
    if comparison == 'B':
        inj = params['t_inj_FT'] if model == 'FT_ref' else params['t_inj_NL']
        add('L0 inject 4 |m> (qubits 1-4, parallel)', idle['qubits'], inj, idle, 0, Counter({'T': 4}), SOURCE_LIT,
            "Litinski App. A / Li: 3+3 cycles, d-independent; physical T gate per |m>")
    add('L1 init qubit 5 |+> (step 1)', idle['qubits'], 0, idle, 0, Counter(), SOURCE_LIT, '0 tau (Litinski)')
    for k in range(11):
        if comparison == 'B':
            add(f'L2.{k + 1}a inject |m> for rotation {k + 1}', idle['qubits'], inj, idle, 0, Counter({'T': 1}),
                SOURCE_LIT, 'serial in the 11-tile layout (Litinski "Caveat")')
        g = Counter({'CNOT': rot[k]['cnot'] * tau}) if rot[k]['cnot'] else Counter()
        add(f'L2.{k + 1} auto-corrected pi/8 rotation {k + 1}', rot[k]['qubits'], tau, rot[k], 0, g, src_rot,
            '1 tau = d cycles: P(x)Z_m surgery (explicit) and Z_m(x)Y_0 (twist not constructed)')
        add(f'L2.{k + 1}b readout |0>,|m> (single-qubit meas.)', idle['qubits'], 0, idle, meas_layer, Counter(),
            SOURCE_LIT, '0 tau; classical processing time not counted')
    add('L3 X measurement of qubits 1-4', idle['qubits'], 0, idle, meas_layer, Counter(), SOURCE_LIT, '0 tau')
    add('L4 logical X frame: T^dag|+> -> T|+>', 0, 0, idle, 0, Counter(), SOURCE_COUNT, 'Pauli frame')
    return rows

In [21]:
# ----------------------------------------------------------------- full resource tables (used by main + comparison)
def msd_resource_tables(D_LIST, dur=None, params=None, verbose=True):
    """Per-stage and summary tables of the Litinski 15-to-1 block for every d in D_LIST, level
    ('tile' | 'physical'), counting variant, time model and comparison A/B.  Litinski uses only
    Clifford + T, so the 'primitive' and 'clifford_t' gate-set rows are identical.
    Returns (stage_df, sum_df, per_d_df)."""
    import pandas as pd, time
    dur = dur or DUR_UNIT
    params = params or DEFAULT_PARAMS
    records, summaries, per_d = [], [], []
    for d in D_LIST:
        t0 = time.time()
        md_ = MSDCostData(d, dur)
        for level in ['tile', 'physical']:
            for data_only in [False, True]:
                for model in ['FT_ref', 'NL_min']:
                    for comp in ['A', 'B']:
                        st = msd_stages(md_, model, params, comp, level=level, data_only=data_only)
                        for gs in ['primitive', 'clifford_t']:
                            records += [dict(r, level=level, gateset=gs, data_only=data_only) for r in st]
                            s = summarize(st)
                            s.update(protocol='Litinski_15to1', d=d, model=model, comparison=comp, level=level,
                                     gateset=gs, data_only=data_only)
                            summaries.append(s)
        per_d.append(dict(d=d, rotation_qubits_min=min(r['qubits'] for r in md_.rot),
                          rotation_qubits_max=max(r['qubits'] for r in md_.rot), idle_qubits=md_.idle['qubits'],
                          rotation_round_lb=max(r['time_lb'] for r in md_.rot),
                          rotation_round_ub=max(r['time_ub'] for r in md_.rot),
                          CNOT_per_round_max=max(r['cnot'] for r in md_.rot), tile_model_rotation_qubits=22 * d * d,
                          isolated_patch_round_lb=md_.round['time_lb'], isolated_patch_round_ub=md_.round['time_ub']))
        if verbose:
            print(f'Litinski d={d}: {time.time() - t0:.1f}s')
    return jsonify_columns(pd.DataFrame(records)), jsonify_columns(pd.DataFrame(summaries)), pd.DataFrame(per_d)

In [22]:
print('library loaded')

library loaded


In [23]:
import itertools
VAL = {}
def record(name, passed, detail=''):
    detail = str(detail).replace('\n', ' ')
    VAL[name] = dict(status='PASS' if passed else 'FAIL', detail=detail)
    print(('PASS ' if passed else 'FAIL ') + name + (f'  | {detail}' if detail else ''))
rng = np.random.default_rng(SEED)

## M1. 行列と回路の整合
Fig. 15 の各列（qubit 1..5 の作用）を図から手で転記した表 `FIG15_TRANSCRIBED` と、式(1) から機械的に生成した回転を比較する。

In [24]:
FIG15_TRANSCRIBED = ['11ZZZ', '1Z1ZZ', '1ZZ1Z', '1ZZZ1', 'Z11ZZ', 'Z1Z1Z', 'Z1ZZ1', 'ZZ11Z', 'ZZ1Z1', 'ZZZ11', 'ZZZZZ']
derived = [''.join('Z' if i in r else '1' for i in range(5)) for r in fig15_rotations()]
record('M1a triorthogonality of G (eq. 2)', all(triorthogonal(G_TRI)), triorthogonal(G_TRI))
# G~ is not literally row-equivalent to G (the row spaces differ): the elimination in [Lit] also reorders
# columns.  Equivalence up to row operations + column permutation: the 16 columns of G are exactly the affine
# hyperplane {c : c_0 = 1}; we check that the 16 columns of G~ are exactly an affine hyperplane {c : f.c = 1}.
def affine_hyperplane_functional(Mx):
    for f in itertools.product((0, 1), repeat=Mx.shape[0]):
        if len({tuple(c) for c in Mx.T}) == 2 ** (Mx.shape[0] - 1) and all(np.dot(f, c) % 2 == 1 for c in Mx.T):
            return f
    return None
fG, fGt = affine_hyperplane_functional(G_TRI), affine_hyperplane_functional(G_TILDE)
record('M1b G~ (eq. 4) is triorthogonal and equivalent to G (eq. 2) up to row operations and a column permutation',
       all(triorthogonal(G_TILDE)) and fG is not None and fGt is not None,
       f'columns of G: hyperplane f={fG}; columns of G~: hyperplane f={fGt}; literal row-equivalence: '
       f'{f2_rank(np.vstack([G_TRI, G_TILDE])) == 5} (column order differs)')
record('M1c M_15-to-1 (eq. 1) = G~ with the first column removed (puncturing)', (G_TILDE[:, 1:] == M15).all())
record('M1d 11 rotations derived from eq. (1) = Fig. 15 (manual transcription)', derived == FIG15_TRANSCRIBED, derived)

PASS M1a triorthogonality of G (eq. 2)  | (True, True, True)
PASS M1b G~ (eq. 4) is triorthogonal and equivalent to G (eq. 2) up to row operations and a column permutation  | columns of G: hyperplane f=(1, 0, 0, 0, 0); columns of G~: hyperplane f=(1, 1, 1, 1, 1); literal row-equivalence: False (column order differs)
PASS M1c M_15-to-1 (eq. 1) = G~ with the first column removed (puncturing)
PASS M1d 11 rotations derived from eq. (1) = Fig. 15 (manual transcription)  | ['11ZZZ', '1Z1ZZ', '1ZZ1Z', '1ZZZ1', 'Z11ZZ', 'Z1Z1Z', 'Z1ZZ1', 'ZZ11Z', 'ZZ1Z1', 'ZZZ11', 'ZZZZZ']


## M2. 理想入力での 5 量子ビット回路

In [25]:
res = run_15to1_5qubit()
nz = {k: v for k, v in res.items() if v[0] > 1e-12}
Tp, Tdp = T_GATE @ KETP, T_GATE.conj() @ KETP
out = nz.get((1, 1, 1, 1), (0, None))[1]
record('M2a ideal inputs: X outcomes of qubits 1-4 are deterministically +1', list(nz) == [(1, 1, 1, 1)] and abs(nz[(1, 1, 1, 1)][0] - 1) < 1e-12)
record('M2b output of qubit 5 = T^dag|+> (not T|+>)', equal_up_to_phase(out, Tdp) and not equal_up_to_phase(out, Tp))
record('M2c X T^dag|+> = e^{-i pi/4} T|+>  (Pauli-frame unification used for both protocols)',
       np.allclose(X @ Tdp, np.exp(-1j * np.pi / 4) * Tp))

PASS M2a ideal inputs: X outcomes of qubits 1-4 are deterministically +1
PASS M2b output of qubit 5 = T^dag|+> (not T|+>)
PASS M2c X T^dag|+> = e^{-i pi/4} T|+>  (Pauli-frame unification used for both protocols)


## M3. 15 量子ビット符号（Fig. 14）

In [26]:
rT = rm15_logical_check('T'); rTd = rm15_logical_check('Tdag')
record('M3a transversal T on |+_L> gives <X_L>=1/sqrt2, <Y_L>=-1/sqrt2 (logical T^dag|+>), Z stabilisers kept',
       abs(rT['X_L'] - 2 ** -.5) < 1e-9 and abs(rT['Y_L'] + 2 ** -.5) < 1e-9 and rT['min_Zstab'] > 1 - 1e-9, rT)
record('M3b transversal T^dag gives logical T|+>', abs(rTd['Y_L'] - 2 ** -.5) < 1e-9, rTd)
bloch_5q = [np.real(np.vdot(out, P @ out)) for P in (X, Y, Z)]
record('M3c 5-qubit circuit output has the same Bloch vector as the 15-qubit logical state',
       np.allclose(bloch_5q, [rT['X_L'], rT['Y_L'], rT['Z_L']], atol=1e-9), np.round(bloch_5q, 6))

PASS M3a transversal T on |+_L> gives <X_L>=1/sqrt2, <Y_L>=-1/sqrt2 (logical T^dag|+>), Z stabilisers kept  | {'X_L': np.float64(0.7071067811865475), 'Y_L': np.float64(-0.7071067811865475), 'Z_L': np.float64(-4.163336342344337e-17), 'min_Zstab': np.float64(0.9999999999999998), 'n_Zstab': 10}
PASS M3b transversal T^dag gives logical T|+>  | {'X_L': np.float64(0.7071067811865475), 'Y_L': np.float64(0.7071067811865475), 'Z_L': np.float64(-4.163336342344337e-17), 'min_Zstab': np.float64(0.9999999999999998), 'n_Zstab': 10}
PASS M3c 5-qubit circuit output has the same Bloch vector as the 15-qubit logical state  | [ 0.707107 -0.707107  0.      ]


## M4. 誤り検出（15 個の π/8 回転のどれかに $Z$ 誤り）

In [27]:
cols15 = [tuple(int(i) for i in np.nonzero(M15[:, j])[0]) for j in range(15)]
def accept(err):
    psi = kron_all([KETP.reshape(2, 1)] * 5).reshape(-1)
    for j, qs in enumerate(cols15):
        psi = z_rotation_diag(5, qs, np.pi / 8 + (np.pi / 2 if j in err else 0)) * psi
    p = 1.0; st = psi
    for q in range(4):
        _, st, pp = measure_pauli(st, pauli_on(5, {q: 'X'}), forced=1); p *= pp
        if pp < 1e-14: return 0.0, None
    o = kron_all([np.array([1, 1]) / np.sqrt(2)] * 4).reshape(-1).conj() @ st.reshape(16, 2)
    return p, o / np.linalg.norm(o)
und = {}
ideal = accept(())[1]
wrong3 = 0
for w in (1, 2, 3):
    und[w] = 0
    for e in itertools.combinations(range(15), w):
        p, o = accept(e)
        if p > 1e-9:
            und[w] += 1
            wrong3 += (w == 3 and not equal_up_to_phase(o, ideal))
record('M4 all 1- and 2-error patterns detected; 35 undetected 3-error patterns, all with a wrong output (output error 35 p^3)',
       und[1] == 0 and und[2] == 0 and und[3] == 35 and wrong3 == 35, f'undetected {und}; wrong outputs among undetected weight-3: {wrong3}')

PASS M4 all 1- and 2-error patterns detected; 35 undetected 3-error patterns, all with a wrong output (output error 35 p^3)  | undetected {1: 0, 2: 0, 3: 35}; wrong outputs among undetected weight-3: 35


## M5. selective π/4（Fig. 17a）と auto-corrected π/8（Fig. 17b）
選んだ 4 種の Pauli 積 $P$（1〜3 量子ビット）とランダムな入力状態について、**全ての測定結果の分岐**で、補正後のデータ状態が目的の回転と一致するか（必要な補正が Pauli $P$ だけか）を確認する。任意の $P$・任意の入力についての証明ではない（線形性により入力は 1 つのランダム状態で十分だが、$P$ は代表例のみ）。

In [28]:
def selective_pi4(P_data, psi, basis):
    k = int(np.log2(len(psi))); n = k + 1; a = k
    st0 = np.kron(psi, KET0)
    Pd = pauli_on(k, P_data)
    target = (np.cos(np.pi / 4) * psi - 1j * np.sin(np.pi / 4) * (Pd @ psi)) if basis == 'X' else psi
    outs = []
    for o1 in (1, -1):
        _, st, p1 = measure_pauli(st0, pauli_on(n, {**P_data, a: 'Y'}), forced=o1)
        if p1 < 1e-14: continue
        for o2 in (1, -1):
            _, st2, p2 = measure_pauli(st, pauli_on(n, {a: basis}), forced=o2)
            if p2 < 1e-14: continue
            anc = (np.array([1, o2]) / np.sqrt(2)) if basis == 'X' else (KET0 if o2 == 1 else np.array([0, 1]))
            dat = st2.reshape(2 ** k, 2) @ anc.conj(); dat /= np.linalg.norm(dat)
            outs.append('I' if equal_up_to_phase(dat, target) else 'P' if equal_up_to_phase(Pd @ dat, target) else 'FAIL')
    return outs
ok = True; table = []
for P in [{0: 'Z'}, {0: 'Z', 1: 'Z'}, {0: 'X', 1: 'Y'}, {0: 'Y', 1: 'Z', 2: 'X'}]:
    k = max(P) + 1
    psi = rng.normal(size=2 ** k) + 1j * rng.normal(size=2 ** k); psi /= np.linalg.norm(psi)
    for basis in 'XZ':
        o = selective_pi4(P, psi, basis); ok &= 'FAIL' not in o
    br = auto_corrected_pi8(P, psi)
    ok &= all(b['correction'] != 'FAIL' for b in br) and abs(sum(b['prob'] for b in br) - 1) < 1e-9
    table += [dict(P=str(P), **{kk: b[kk] for kk in ['PZ', 'ZY', 'anc_basis', 'anc', 'Xm', 'correction']}, prob=round(b['prob'], 4)) for b in br]
record('M5 selective pi/4 and auto-corrected pi/8 (4 sample Pauli products, random input): every branch correct up to a Pauli P correction', ok)
pd.DataFrame(table).head(16)

PASS M5 selective pi/4 and auto-corrected pi/8 (4 sample Pauli products, random input): every branch correct up to a Pauli P correction


,P,PZ,ZY,anc_basis,anc,Xm,correction,prob
0,{0: 'Z'},1,1,Z,1,1,I,0.0625
1,{0: 'Z'},1,1,Z,1,-1,P,0.0625
2,{0: 'Z'},1,1,Z,-1,1,P,0.0625
3,{0: 'Z'},1,1,Z,-1,-1,I,0.0625
4,{0: 'Z'},1,-1,Z,1,1,I,0.0625
5,{0: 'Z'},1,-1,Z,1,-1,P,0.0625
6,{0: 'Z'},1,-1,Z,-1,1,P,0.0625
7,{0: 'Z'},1,-1,Z,-1,-1,I,0.0625
8,{0: 'Z'},-1,1,X,1,1,I,0.0625
9,{0: 'Z'},-1,1,X,1,-1,P,0.0625


## M6. tile 数・time step の再計数
(a) 格子転記 `FIG17C_*` から数える。(b) [Lit] 本文の記述（「4 qubits + 1 output」「左の 2×2 ブロックは $|m\rangle$ と $|0\rangle$ 用」「2 tiles は多体測定のアンシラ」「11 回転 × 1τ」）から独立に数える。

In [29]:
sched = tile_schedule(); n_a = len(tiles_used(sched)); tau_a = sum(t for _, t, _ in sched)
n_b = 4 + 1 + 2 * 2 + 2; tau_b = 11 * 1
record('M6 tiles = 11 and time steps = 11 (two independent countings), space-time = 121 d^3',
       n_a == n_b == 11 and tau_a == tau_b == 11, dict(grid=(n_a, tau_a), text=(n_b, tau_b)))
for dd in [3, 5, 7, 9]:
    s = summarize(msd_stages(MSDCostData(dd), 'FT_ref', DEFAULT_PARAMS, 'A', level='tile', data_only=True))
    s2 = summarize(msd_stages(MSDCostData(dd), 'FT_ref', DEFAULT_PARAMS, 'A', level='tile', data_only=False))
    record(f'M6 d={dd}: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)',
           s['V_qubit_round'] == 121 * dd ** 3 and s2['V_qubit_round'] == 242 * dd ** 3)

PASS M6 tiles = 11 and time steps = 11 (two independent countings), space-time = 121 d^3  | {'grid': (11, 11), 'text': (11, 11)}


PASS M6 d=3: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)


PASS M6 d=5: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)


PASS M6 d=7: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)


PASS M6 d=9: tile model reproduces 121 d^3 (data) and 242 d^3 (incl. measurement qubits)


## M7. 物理レベルの格子手術（stim、ノイズなし）
2 つの $d\times d$ 回転表面符号パッチを $|+_L\rangle$ に準備し、間の 1 tile（幅 $d$）を $|+\rangle$ で初期化して合併矩形の安定化子を 1 ラウンド測定する。$\bar Z_1\bar Z_2$ の期待値が合併前 0 → 合併後 ±1 となり、その符号が合併で新たに測った $Z$ 型チェックの積と一致することを確認する（[Lit] App. A の手順）。

In [30]:
for dd in [3, 5, 7]:
    for seed in range(3):
        r = lattice_surgery_zz_check(dd, seed=seed)
        record(f'M7 d={dd} seed={seed}: Z1Z2 random before, deterministic after merge, equals product of merged Z checks',
               r['exp_before'] == 0 and abs(r['exp_after']) == 1 and r['consistent'] and r['O_in_span'],
               f"after={r['exp_after']}, qubits={r['total_qubits']}")
    md_ = MSDCostData(dd)
    record(f'M7 d={dd}: one gap tile adds 2d^2+1 qubits to the merged region (used for the ancilla strip)', md_.gap_tile == 2 * dd * dd + 1, md_.gap_tile)

PASS M7 d=3 seed=0: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=53
PASS M7 d=3 seed=1: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=53
PASS M7 d=3 seed=2: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=-1, qubits=53
PASS M7 d=3: one gap tile adds 2d^2+1 qubits to the merged region (used for the ancilla strip)  | 19
PASS M7 d=5 seed=0: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=149
PASS M7 d=5 seed=1: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=149
PASS M7 d=5 seed=2: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=149


PASS M7 d=5: one gap tile adds 2d^2+1 qubits to the merged region (used for the ancilla strip)  | 51
PASS M7 d=7 seed=0: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=-1, qubits=293
PASS M7 d=7 seed=1: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=-1, qubits=293
PASS M7 d=7 seed=2: Z1Z2 random before, deterministic after merge, equals product of merged Z checks  | after=1, qubits=293


PASS M7 d=7: one gap tile adds 2d^2+1 qubits to the merged region (used for the ancilla strip)  | 99


## M8. パッチの量子ビット数・CNOT 数と換算の一貫性

In [31]:
ok = True
for dd in range(3, 16, 2):
    m = MSDCostData(dd)
    ok &= m.sq_patch == 2 * dd * dd - 1 and m.tall_patch == 4 * dd * dd - 1 and m.cnot_sq == 4 * dd * (dd - 1)
    phys = summarize(msd_stages(m, 'FT_ref', DEFAULT_PARAMS, 'A', level='physical'))
    ok &= phys['peak_qubits'] == 22 * dd * dd - 6 and phys['V_qubit_round'] == 11 * dd * (22 * dd * dd - 6)
record('M8 rotated patch = 2d^2-1 qubits, 4d(d-1) CNOT/round; explicit physical model: 22d^2-6 qubits per rotation (tile model 22d^2), V = 11 d (22d^2-6)', ok)
m = MSDCostData(5)
record('M8 surface-code round time bounds from the shared scheduler (lb=6: init+4 CNOT+meas)', m.round['time_lb'] == 6, m.round)

PASS M8 rotated patch = 2d^2-1 qubits, 4d(d-1) CNOT/round; explicit physical model: 22d^2-6 qubits per rotation (tile model 22d^2), V = 11 d (22d^2-6)
PASS M8 surface-code round time bounds from the shared scheduler (lb=6: init+4 CNOT+meas)  | {'time_lb': 6.0, 'time_ub': 12}


## M9. Fig. 17c の 11 個の多パッチ測定の物理レベル構成と検証

各回転 $k$ で測る演算子は $O_k=\bigotimes_{i\in P_k}\bar Z_i\otimes\bar Z_m$（$P_k$ は式(1)の列）。補助帯を $|+\rangle$ に初期化し、新しい検査 $N$ を測ったときに：
* V1 $N$ が互いに可換、V2 各パッチが単独で $k=1$・距離 $d$、
* V3 $O_k\in\mathrm{span}(N_Z)+\mathrm{span}(G_{0,Z})$ で、測られる $Z$ 論理の組合せはちょうど 1 個、V4 $X$ 論理は 1 個も測られない、
* V5 $O_k$ と可換な全ての論理演算子（全 $\bar Z_i$、非参加パッチの $\bar X_j$、参加パッチ対の $\bar X_i\bar X_{i'}$）が $N$ と可換な代表元をもつ（保存される）、
* 合併符号 $G_1=\langle N,\ G_0\text{ の }N\text{ 可換部分群}\rangle$ の論理量子ビット数が 5、距離 $\ge d$（MILP）、

を厳密に確認する。さらに stim で、合併前は $O_k$ がランダム、合併 1 ラウンド後は決定的で、その値が測定結果の積から予言どおりであることを確認する。

In [32]:
import time
rot_sets = [tuple(sorted({'q%d' % (i + 1) for i in r})) + ('m',) for r in fig15_rotations()]
for dd in [3, 5, 7]:
    t0 = time.time(); allv = True; dists = set()
    for P in rot_sets:
        r = verify_multipatch(dd, P, distance=(dd <= 5))
        allv &= r['valid'] and r['V2_patches_k1']
        if dd <= 5:
            allv &= r['V2_patch_distance_ok']; dists.add(r['V6_merged_distance'])
    record(f'M9 d={dd}: all 11 multi-patch measurements P(x)Z_m are valid lattice surgeries (exact F2 criteria'
           + (', merged-code distance >= d' if dd <= 5 else '') + ')', allv,
           (f'merged (d_X, d_Z) = {sorted(dists)}; ' if dists else 'distance not computed at this d; ') + f'{time.time() - t0:.0f}s')
for dd in [3, 5]:
    ok = all(stim_multipatch_check(dd, P, seed=s)['consistent'] for P in rot_sets for s in range(2))
    record(f'M9 d={dd}: stim - O_k random before, deterministic after one merged round, equal to predicted product (11 rotations x 2 seeds)', ok)
ok = True
for dd in [3, 5, 7]:
    for P in rot_sets:
        a, res = rotation_measured_checks(dd, P)
        b, pats, own = rotation_measured_checks_fast(dd, P)
        qs = sorted(own); idx = {q: i for i, q in enumerate(qs)}; n = len(qs)
        for t in 'XZ':
            A = np.array([_vec(c[1], idx, n) for c in a if c[0] == t]); Bm = np.array([_vec(c[1], idx, n) for c in b if c[0] == t])
            ok &= _rank(A) == _rank(Bm) == _rank(np.vstack([A, Bm]))
        ok &= len(a) == len(b) and max(len(c[1]) for c in b) <= 4
record('M9 fast measured-set construction (used for d up to 15) = verified construction (d=3,5,7); all checks weight <= 4', ok)

PASS M9 d=3: all 11 multi-patch measurements P(x)Z_m are valid lattice surgeries (exact F2 criteria, merged-code distance >= d)  | merged (d_X, d_Z) = [(3, 3), (4, 3)]; 6s


PASS M9 d=5: all 11 multi-patch measurements P(x)Z_m are valid lattice surgeries (exact F2 criteria, merged-code distance >= d)  | merged (d_X, d_Z) = [(5, 5), (8, 5)]; 70s


PASS M9 d=7: all 11 multi-patch measurements P(x)Z_m are valid lattice surgeries (exact F2 criteria)  | distance not computed at this d; 3s


PASS M9 d=3: stim - O_k random before, deterministic after one merged round, equal to predicted product (11 rotations x 2 seeds)


PASS M9 d=5: stim - O_k random before, deterministic after one merged round, equal to predicted product (11 rotations x 2 seeds)


PASS M9 fast measured-set construction (used for d up to 15) = verified construction (d=3,5,7); all checks weight <= 4


In [33]:
n_pass = sum(v['status'] == 'PASS' for v in VAL.values())
print(f'{n_pass} PASS, {len(VAL) - n_pass} FAIL')
json.dump(dict(notebook='MSD_15to1_validation', results=VAL,
               not_verified=['stabilizer-level layout of the Z_m (x) Y_0 twist-based seam (O(d) measurement qubits not counted)',
                             'merged-code distance verified only for d = 3, 5 (validity criteria also for d = 7)',
                             'classical processing latency', 'Fig. 18 variant (extra 2x2 blocks) not modelled']),
          open(f'{RESULTS}/validation_msd.json', 'w'), indent=1)
pd.DataFrame([(k, v['status']) for k, v in VAL.items()], columns=['check', 'status'])

37 PASS, 0 FAIL


,check,status
0,M1a triorthogonality of G (eq. 2),PASS
1,M1b G~ (eq. 4) is triorthogonal and equivalent...,PASS
2,M1c M_15-to-1 (eq. 1) = G~ with the first colu...,PASS
3,M1d 11 rotations derived from eq. (1) = Fig. 1...,PASS
4,M2a ideal inputs: X outcomes of qubits 1-4 are...,PASS
5,M2b output of qubit 5 = T^dag|+> (not T|+>),PASS
6,M2c X T^dag|+> = e^{-i pi/4} T|+> (Pauli-fram...,PASS
7,M3a transversal T on |+_L> gives <X_L>=1/sqrt2...,PASS
8,M3b transversal T^dag gives logical T|+>,PASS
9,M3c 5-qubit circuit output has the same Bloch ...,PASS
